# Мониторинг фин. эффекта по Excel

```
e_fee        = p_fu×100k + p_court×15k
expected_psr = precision × Σ_I (СуммаОсновногоДолгаКВыплате × k + e_fee)
cost         = Σ_I СуммаПлатежа
net          = expected_psr − cost
```

`I` (финэффект): `РезультатПроверки=1` ∧ `Заключено соглашение=1` ∧ `Выплата по модели=1`.  
ОД × k → оценка ПСР; cost = **СуммаПлатежа**. `PRECISION` — вручную.  
Ретро-доли `p_pret`/`p_fu`/`p_court`, `k`, `psr_share` — за **последние 2 года** (`PAYMENT_ORDER_DATE_TIME`).  
Доли соглашений: with_model (без фильтра соглашения) vs without_model.  
HTML-отчёт: `data/fin_effect_report.html`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
for _p in (SRC, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

NOTEBOOK_DIR = PROJECT_ROOT / "monitoring" / "fin_effects"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)

In [ ]:
from IPython.display import display

from querulus.fin_effect.excel_monitoring import (
    compare_agreement_pretension_by_model,
    default_demo_priors,
    estimate_monitoring_effect,
    extrapolate_to_year,
    format_sensitivity_table,
    format_summary_dict,
    infer_model_start,
    load_excel,
    load_retro_priors,
    save_retro_priors,
    sensitivity_table,
    write_synthetic_claims_excel,
)
from querulus.fin_effect.monitoring_report import write_monitoring_html

GENERATE_SYNTHETIC = True
EXCEL_PATH = DATA_DIR / "querulus_claims_synthetic.xlsx"
# EXCEL_PATH = Path("/home/jovyan/old_home/querulus/monitoring/fin_effects/data/claims_prod.xlsx")

PRIORS_PATH = DATA_DIR / "retro_priors.json"
REPORT_HTML = DATA_DIR / "fin_effect_report.html"
# ОД → ×k для ПСР; СуммаПлатежа → cost / net
OD_COL = "СуммаОсновногоДолгаКВыплате"
COST_COL = "СуммаПлатежа"

RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
LOOKBACK_YEARS = 2.0  # окно для k, psr_share, p_pret/p_fu/p_court

PRECISION = 0.49  # вручную 0..1; None → из preds_cf

MODEL_START = None  # None → min даты вызова модели из Excel
AS_OF = None

if GENERATE_SYNTHETIC and not Path(EXCEL_PATH).exists():
    write_synthetic_claims_excel(EXCEL_PATH, n_rows=300)
    print("synthetic written", EXCEL_PATH)

if not PRIORS_PATH.exists():
    save_retro_priors(default_demo_priors(), PRIORS_PATH)
    print("demo priors written", PRIORS_PATH)

df = load_excel(EXCEL_PATH)
print("shape", df.shape)
print("RETRO_PARQUET exists", Path(RETRO_PARQUET).exists())
print("LOOKBACK_YEARS", LOOKBACK_YEARS)
print("PRECISION", PRECISION)
print("MODEL_START override", MODEL_START)
print("MODEL_START from excel", infer_model_start(df).date())

In [ ]:
import pandas as pd
from querulus.fin_effect.excel_monitoring import RetroPriors, compute_retro_priors

if RETRO_PARQUET is not None and Path(RETRO_PARQUET).exists():
    retro = pd.read_parquet(RETRO_PARQUET)
    print("retro shape", retro.shape)
    print("TARGET_FREQ" in retro.columns, "cols sample", list(retro.columns)[:8])
    priors = compute_retro_priors(
        retro,
        precision=PRECISION,
        lookback_years=LOOKBACK_YEARS,
    )
    save_retro_priors(priors, PRIORS_PATH)
    print("priors from parquet (last", LOOKBACK_YEARS, "y) →", PRIORS_PATH)
    print(
        "window",
        priors.window_start,
        "…",
        priors.window_end,
        "date_col",
        priors.date_column,
        "n",
        priors.n_rows,
        "psr_share",
        round(priors.psr_share, 4),
    )
else:
    priors = load_retro_priors(PRIORS_PATH)
    if PRECISION is not None:
        priors = RetroPriors(
            precision=float(PRECISION),
            k=priors.k,
            p_pret=priors.p_pret,
            p_fu=priors.p_fu,
            p_court=priors.p_court,
            fu_fee=priors.fu_fee,
            court_fee=priors.court_fee,
            lookback_years=priors.lookback_years,
            date_column=priors.date_column,
            window_start=priors.window_start,
            window_end=priors.window_end,
            n_rows=priors.n_rows,
            n_pos=priors.n_pos,
            psr_share=priors.psr_share,
        )
        save_retro_priors(priors, PRIORS_PATH)
    print("priors from json (parquet missing):", RETRO_PARQUET)

print(priors)
print("e_fee", round(priors.expected_fee(), 2))
print(
    "p_fu / p_court / p_pret",
    round(priors.p_fu, 4),
    round(priors.p_court, 4),
    round(priors.p_pret, 4),
)

In [ ]:
effect = estimate_monitoring_effect(
    df, priors, od_col=OD_COL, cost_col=COST_COL
)
summary = {
    "od_column": effect.od_column,
    "cost_column": effect.cost_column,
    "n_intervention": effect.n_intervention,
    "sum_od": round(effect.sum_od, 2),
    "sum_paid": round(effect.sum_paid, 2),
    "e_fee": round(effect.e_fee, 2),
    "expected_psr": round(effect.expected_psr, 2),
    "cost": round(effect.cost, 2),
    "net": round(effect.net, 2),
    "psr_share_retro": round(priors.psr_share, 4),
    "retro_window": f"{priors.window_start} … {priors.window_end}",
}
display(format_summary_dict(summary))

sens = format_sensitivity_table(
    sensitivity_table(df, priors, od_col=OD_COL, cost_col=COST_COL)
)
display(sens)

annual = extrapolate_to_year(
    effect, df, model_start=MODEL_START, as_of=AS_OF, year_days=365.0
)
display(format_summary_dict(annual))

seg = compare_agreement_pretension_by_model(df)
for col in ("agreement_share", "pretension_share"):
    seg[col] = (seg[col] * 100).round(2)
display(seg)

report_path = write_monitoring_html(
    effect,
    REPORT_HTML,
    annual=format_summary_dict(annual),
    segments=seg,
)
print("HTML report →", report_path)